<a href="https://colab.research.google.com/github/ChJazhiel/TEC_CCM_2026/blob/main/2D_Ejercicio_Semana8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicios de Selección de Variables

En este notebook practicarás distintos métodos de selección de variables:
- Filtros univariantes
- Eliminación recursiva con validación cruzada (RFECV)
- Comparación de métodos filter, wrapper y embedded

Cada ejercicio tiene instrucciones en celdas Markdown y una celda de código para que desarrolles tu solución.

## Ejercicio 1: Selección de variables con filtro + métodos embebidos (E-commerce Churn)

Comparar una técnica de filtro (`f_classif`) con dos métodos embebidos
basados en regularización (**Lasso** y **Ridge**), variando el hiperparámetro `alpha`
de penalización en el rango (0, 1].

**Tareas:**
1. Cargar el dataset `ecommerce_customer_churn_dataset.csv` y preprocesarlo
   (imputación + OneHot + escalado).
2. Dividir en train/test estratificado por `Churned`.
3. Aplicar tres métodos de selección:
   - **Filtro:** `SelectKBest` con `f_classif` (top-15).
   - **Lasso:** `Lasso(alpha=...)` para varios valores de alpha en {0.001, 0.01, 0.1, 0.5}.
     Se seleccionan las variables con coeficiente ≠ 0. Si selecciona menos de 15,
     se completan con las de mayor |coef|.
   - **Ridge:** `Ridge(alpha=...)` para los mismos valores. Ridge no anula coeficientes,
     por lo que se eligen las **top-15 por |coef|**.
4. Entrenar una `Random Forest` con 100 árboles y 20 muestras por hoja sobre cada subconjunto y comparar
   Accuracy, F1 y ROC-AUC.
5. Analizar cómo cambia el número de variables seleccionadas y el rendimiento
   al aumentar `alpha` (mayor penalización).

**Objetivo:** Haber entrenado y evaluado todos los modelos, y explicar
cómo `alpha` afecta al compromiso sesgo-varianza y al rendimiento final.

In [ ]:
# ============================================================
# EJERCICIO 1 (E-COMMERCE): F_CLASSIF + LASSO + RIDGE
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import Lasso, Ridge, LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ---------- 1) Cargar datos ----------
df = pd.read_csv("ecommerce_customer_churn_dataset.csv")
print("Shape:", df.shape)

X = df.drop(columns=["Churned"])
y = df["Churned"].astype(int)

num_features = X.select_dtypes(include=np.number).columns.tolist()
cat_features = X.select_dtypes(include="object").columns.tolist()

# ---------- Continua a partir de aqui ------------

## Ejercicio 2: Importancias por Entropía + RFE (E-commerce Churn)

Usar dos enfoques basados en árboles para seleccionar variables
sobre el dataset de churn:

1. **Entropía:** `RandomForestClassifier(criterion='entropy')` → `feature_importances_`.
2. **RFE:** `Recursive Feature Elimination` con un `RandomForestClassifier` como
   estimador.

En ambos casos seleccionamos **15 variables** y comparamos el rendimiento frente
al modelo completo.

**Tareas:**
1. Entrenar un Random Forest con `criterion='entropy'` (n_estimators=300).
2. Extraer las **top-15 variables** por importancia de entropía.
3. Aplicar `RFE` con RF como estimador y `n_features_to_select=15`.
4. Comparar las 15 variables seleccionadas por cada método.
5. Entrenar un RF final con cada subconjunto y comparar Accuracy, F1 y ROC-AUC
   frente al baseline (todas las variables).

**Objetivo:** Haber identificado las 15 variables más relevantes por cada
método, medido su rendimiento y discutido coincidencias y diferencias.

In [ ]:
# ============================================================
# EJERCICIO 2 (E-COMMERCE): ENTROPÍA + RFE (top-15)
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE

K = 15  # <-- 15 variables

# ---------- 1) Random Forest con criterion='entropy' ----------
rf_entropy = RandomForestClassifier(
    n_estimators=300,
    criterion="entropy",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_entropy.fit(X_train_pre, y_train)
y_pred_full  = rf_entropy.predict(X_test_pre)
y_proba_full = rf_entropy.predict_proba(X_test_pre)[:, 1]

print("=== RF Entropía (todas las variables) ===")
print("Accuracy:", round(accuracy_score(y_test, y_pred_full), 4))
print("F1      :", round(f1_score(y_test, y_pred_full), 4))
print("ROC_AUC :", round(roc_auc_score(y_test, y_proba_full), 4))

# ---------- 2) Importancias por entropía (top-15) ----------
entropy_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_entropy.feature_importances_
}).sort_values("importance", ascending=False)

top15_entropy = entropy_df.head(K)["feature"].tolist()
print(f"\nTop-{K} por Entropía:")
print(entropy_df.head(K).to_string(index=False))

plt.figure(figsize=(9, 6))
sns.barplot(data=entropy_df.head(K), x="importance", y="feature", palette="viridis")
plt.title(f"Random Forest (entropy) - Top {K} variables")
plt.tight_layout()
plt.show()

# ----------- Continua a partir de aqui ------------